### Minicurso Sistemas Multi-agente com LangGraph

## Chamada à LLM

Moacir Antonelli Ponti - 2025


---

Criar arquivo .env:

`echo "OPENAI_API_KEY=....." > .env`

In [ ]:
!echo "OPENAI_API_KEY=suachavedaopenaivamosdardinheiroparaosamaltman" > .env

In [9]:
!cat .env_2

OPENAI_API_KEY=suachavedaopenaivamosdardinheiroparaosamaltman


In [ ]:
#%%capture
#!pip install langchain-openai

In [1]:
from typing import TypedDict, List
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
class AgentState(TypedDict):
    # ao invocar grafo vamos listando as mensagens humanas
    messages: List[HumanMessage]

# initialize language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

In [3]:
def process(state: AgentState) -> AgentState:
    # input is the question
    response = llm.invoke(input=state["messages"])
    print(f"\nAI: {response.content}")
    return state

graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)

agent = graph.compile()

user_input = input("Enter: ")
while user_input != "exit":
    agent.invoke({"messages" : [HumanMessage(content=user_input)]})
    user_input = input("Enter: ")



AI: A capital da Paraíba é João Pessoa.

AI: A banda Raimundos é originária de Brasília, capital do Brasil. Formada em 1987, a banda se destacou por seu estilo que mistura rock com influências do punk e do hardcore, além de elementos da música popular brasileira.


## Exercício:

Crie um agente que analise se uma frase é um "elogio" um "crítica" ou "neutra".
Armazenando no estado do agente uma variável `classe` essa informação

In [ ]:
class AgentState(TypedDict):
    messages: List[HumanMessage]
    classe: str  # "elogio" ou "critica"

llm = ChatOpenAI(model="gpt-4o-mini")

def analisa(state: AgentState) -> AgentState:
    texto = state["messages"][-1].content

    # gera lista de strings para pegar todas as mensagens do contexto
    # texto = [msg.content for msg in state["messages"]]

    prompt = f"""
    Classifique a frase como "elogio" (tom positivo), 
    "critica" (tom negativo ou frustrado),
    ou "neutro" (quando não for possível dizer se é um elogio ou crítica com certeza)
    Responda somente uma palavra: elogio, neutro ou critica.

    Frase: {texto}
    """
    response = llm.invoke(input=prompt).content.strip()

    if "neutro" in response:
        state['classe'] = "neutro"
    elif "critica" in response:
        state['classe'] = "critica"
    elif "elogio" in response:
        state['classe'] = "elogio"
    else:
        state['classe'] = "indeterminado"

    return state


In [9]:
graph = StateGraph(AgentState)
graph.add_node("analisa", analisa)
graph.add_edge(START, "analisa")
graph.add_edge("analisa", END)

agent = graph.compile()

while True:
    texto = input("Você: ")
    if texto == "exit":
        break

    entrada = {"messages": [HumanMessage(content=texto)], "classe": ""}
    res = agent.invoke(entrada)
    print(f"Texto: {res['messages'][-1].content}, Sentimento: {res['classe']}")


Texto: estou muito feliz, Sentimento: elogio
Texto: estou passando raiva, Sentimento: indeterminado


In [ ]:
graph = StateGraph(AgentState)
graph.add_node("analisa", analisa)
graph.add_edge(START, "analisa")
graph.add_edge("analisa", END)

agent = graph.compile()

#cria uma lista de mensagens
message_content = []
while True:
    texto = input("Você: ")
    if texto == "exit":
        break

    message_content.append(HumanMessage(content=texto))
    
    print(f'Conteúdo completo: {message_content}')

    entrada = {"messages": message_content, "classe": ""}
    res = agent.invoke(entrada)
    print(f"Texto: {res['messages'][-1].content}, Sentimento: {res['classe']}")


Conteúdo completo: [HumanMessage(content='é uma lista', additional_kwargs={}, response_metadata={})]
Texto: é uma lista, Sentimento: neutro
Conteúdo completo: [HumanMessage(content='é uma lista', additional_kwargs={}, response_metadata={}), HumanMessage(content='entao agora', additional_kwargs={}, response_metadata={})]
Texto: entao agora, Sentimento: neutro
Conteúdo completo: [HumanMessage(content='é uma lista', additional_kwargs={}, response_metadata={}), HumanMessage(content='entao agora', additional_kwargs={}, response_metadata={}), HumanMessage(content='nao sei o que fazer', additional_kwargs={}, response_metadata={})]
Texto: nao sei o que fazer, Sentimento: neutro
Conteúdo completo: [HumanMessage(content='é uma lista', additional_kwargs={}, response_metadata={}), HumanMessage(content='entao agora', additional_kwargs={}, response_metadata={}), HumanMessage(content='nao sei o que fazer', additional_kwargs={}, response_metadata={}), HumanMessage(content='já sei, vou extrair cada con